# Enrollee-Bot — RAG experiments (FCS HSE admission chatbot)

Reproducible experiments for the project report: data preprocessing, chunking,
embedder selection, reranking, and generation + its evaluation.

**How to run:** open in Google Colab → `Runtime → Change runtime type → GPU (T4)` →
`Runtime → Run all`. Everything below uses only **free** models (no paid API).
Full run takes ~20–40 min on a free T4.

The only required input file is `data/fcs_hse_qa_dataset.json` (the gold set, 50 Q&A).
The knowledge corpus `data/text.txt` is rebuilt automatically from the public HSE
website; if scraping is unavailable, a fallback corpus is built from the gold answers
(this is clearly reported, because it makes retrieval easier and inflates the numbers).

In [ ]:
!pip install -q sentence-transformers faiss-cpu langchain langchain-text-splitters \
                 langchain-experimental transformers accelerate pymorphy3 \
                 beautifulsoup4 requests pandas numpy tqdm scikit-learn

In [ ]:
import os, re, json, time, unicodedata, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

DATA = Path("data"); DATA.mkdir(exist_ok=True)
QA_PATH = DATA / "fcs_hse_qa_dataset.json"
assert QA_PATH.exists(), "Put fcs_hse_qa_dataset.json into ./data/ (upload it in Colab)."
RESULTS = {}

## 1. Knowledge corpus

The original `text.txt` was lost, so we rebuild it from the public FCS HSE admission
pages (the same official sources the bot is meant to serve). Scraping uses
`requests` + `BeautifulSoup`. Pages that fail are skipped. If too little text is
collected, we fall back to a corpus assembled from the gold reference answers and
print a warning — that path is only a safety net, not the intended setup.

In [ ]:
import requests
from bs4 import BeautifulSoup

URLS = [
    "https://cs.hse.ru/abitur/",
    "https://cs.hse.ru/abitur/ba",
    "https://cs.hse.ru/abitur/ma",
    "https://cs.hse.ru/spravochnik",
    "https://ba.hse.ru/",
    "https://www.hse.ru/ba/ami/",
    "https://www.hse.ru/ba/se/",
    "https://www.hse.ru/ba/data/",
    "https://www.hse.ru/ba/sec/",
]
HEADERS = {"User-Agent": "Mozilla/5.0 (research crawler; admission FAQ corpus)"}

def fetch_text(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        main = soup.find("div", class_=re.compile("content|main|post", re.I)) or soup.body or soup
        text = main.get_text(separator="\n")
        return url, text
    except Exception as e:
        print("  skip", url, "->", type(e).__name__)
        return url, ""

docs = []
for u in URLS:
    url, t = fetch_text(u)
    if len(t.strip()) > 400:
        docs.append((url, t))
        print(f"  ok  {url}  ({len(t)} chars)")

scraped = "\n\n".join(t for _, t in docs)
print(f"\nscraped: {len(docs)} pages, {len(scraped)} chars")

if len(scraped) < 8000:
    print("\n[!] WARNING: too little scraped text. Falling back to a corpus built from the")
    print("    gold answers. Retrieval numbers from this fallback are OPTIMISTIC — report it.")
    qa_tmp = json.loads(QA_PATH.read_text(encoding="utf-8"))
    scraped = "\n\n".join(f"{x['context']}. {x['answer']}" for x in qa_tmp)
    CORPUS_SOURCE = "fallback (gold answers)"
else:
    CORPUS_SOURCE = f"scraped public HSE pages ({len(docs)} pages)"

(DATA / "text.txt").write_text(scraped, encoding="utf-8")
raw_corpus = scraped
print("corpus source:", CORPUS_SOURCE)

## 2. Datasets — basic EDA

`text.txt` is the knowledge corpus; `fcs_hse_qa_dataset.json` is the gold benchmark
(50 question / reference-answer pairs). Below we print descriptive statistics used in
the report: corpus size, category balance, and question/answer length.

In [ ]:
qa = json.loads(QA_PATH.read_text(encoding="utf-8"))
print("CORPUS")
print("  chars:", len(raw_corpus), "| words:", len(raw_corpus.split()))
print()
print("GOLD QA SET")
print("  pairs:", len(qa))
cat = pd.Series([x["category"] for x in qa]).value_counts()
print("  categories:", cat.shape[0])
print(cat.to_string())
qw = [len(re.findall(r"\w+", x["question"])) for x in qa]
aw = [len(re.findall(r"\w+", x["answer"])) for x in qa]
print(f"\n  question words: min {min(qw)} / median {int(np.median(qw))} / max {max(qw)} / mean {np.mean(qw):.1f}")
print(f"  answer words:   min {min(aw)} / median {int(np.median(aw))} / max {max(aw)} / mean {np.mean(aw):.1f}")
RESULTS["eda"] = {"corpus_chars": len(raw_corpus), "corpus_words": len(raw_corpus.split()),
                  "n_qa": len(qa), "n_categories": int(cat.shape[0]),
                  "category_counts": cat.to_dict(),
                  "q_words_mean": float(np.mean(qw)), "a_words_mean": float(np.mean(aw)),
                  "corpus_source": CORPUS_SOURCE}

## 3. Preprocessing

Five cumulative configurations, from raw text to a fully cleaned version.
Fixed embedder `intfloat/multilingual-e5-small`, fixed chunking (500 chars / 10%).
Metric: Hit Rate@5 (a question counts as a hit if at least one of the top-5 chunks
shares **>= 2 lemmas** with the reference answer; lemmatisation via `pymorphy3`).

In [ ]:
def clean_ws(t):
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)                 # de-hyphenate line breaks
    t = re.sub(r"(?<=[а-яёa-z,])\n(?=[а-яёa-z])", " ", t)  # join mid-sentence breaks
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def strip_headers(t):
    pats = [r"^\s*НИУ ВШЭ\s*$", r"^\s*стр\.?\s*\d+.*$", r"^\s*\d+\s*/\s*\d+\s*$",
            r"^\s*©.*$", r"^\s*Факультет компьютерных наук\s*$",
            r"^\s*Мы используем файлы cookies.*$"]
    for p in pats:
        t = re.sub(p, "", t, flags=re.MULTILINE | re.IGNORECASE)
    return re.sub(r"\n{3,}", "\n\n", t).strip()

def norm_unicode(t):
    t = unicodedata.normalize("NFKC", t)
    for a, b in {"«": '"', "»": '"', "“": '"', "”": '"', "—": "-", "–": "-",
                 "\u2212": "-", "\xa0": " ", "\u2009": " "}.items():
        t = t.replace(a, b)
    return t

PREPROCESS = {
    "baseline":     lambda t: t,
    "+ whitespace": clean_ws,
    "+ headers":    lambda t: strip_headers(clean_ws(t)),
    "+ unicode":    lambda t: norm_unicode(strip_headers(clean_ws(t))),
}
for k, fn in PREPROCESS.items():
    print(f"{k:14s} {len(fn(raw_corpus))} chars")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

def chunk_fixed(text, size=500, overlap=0.1):
    s = CharacterTextSplitter(separator="", chunk_size=size, chunk_overlap=int(size*overlap))
    return [c for c in s.split_text(text) if c.strip()]

def chunk_recursive(text, size=500, overlap=0.1):
    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=int(size*overlap),
                                       separators=["\n\n", "\n", ". ", " ", ""])
    return [c for c in s.split_text(text) if c.strip()]

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

# Bi-encoder + FAISS (cosine via normalised inner product).
# e5 models need 'query:'/'passage:' prefixes; bge-m3 and others do not.
class Index:
    def __init__(self, model_name):
        self.name = model_name
        self.is_e5 = "e5" in model_name.lower()
        self.model = SentenceTransformer(model_name, device=DEVICE)
        self.chunks = []
    def _passages(self, xs): return [f"passage: {x}" for x in xs] if self.is_e5 else xs
    def _queries(self, xs):  return [f"query: {x}"   for x in xs] if self.is_e5 else xs
    def build(self, chunks):
        self.chunks = chunks
        emb = self.model.encode(self._passages(chunks), normalize_embeddings=True,
                                batch_size=32, show_progress_bar=False).astype("float32")
        self.idx = faiss.IndexFlatIP(emb.shape[1]); self.idx.add(emb)
    def search(self, q, k=5):
        qv = self.model.encode(self._queries([q]), normalize_embeddings=True).astype("float32")
        s, i = self.idx.search(qv, k)
        return list(zip(i[0].tolist(), s[0].tolist()))

In [ ]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()
_cache = {}
def lemmas(text):
    if text in _cache: return _cache[text]
    out = {morph.parse(t)[0].normal_form for t in re.findall(r"\w+", text.lower()) if len(t) > 2}
    _cache[text] = out
    return out

def is_hit(chunk, ref, min_match=2):
    return len(lemmas(chunk) & lemmas(ref)) >= min_match

def hit_rate(index, k=5):
    return np.mean([any(is_hit(index.chunks[i], it["answer"]) for i, _ in index.search(it["question"], k))
                    for it in qa])

def mrr(index, k=10):
    s = 0.0
    for it in qa:
        for rank, (i, _) in enumerate(index.search(it["question"], k), 1):
            if is_hit(index.chunks[i], it["answer"]):
                s += 1.0 / rank; break
    return s / len(qa)

In [ ]:
EMB_SMALL = "intfloat/multilingual-e5-small"
rows = []
for name, fn in PREPROCESS.items():
    chunks = chunk_fixed(fn(raw_corpus), 500, 0.1)
    idx = Index(EMB_SMALL); idx.build(chunks)
    rows.append({"configuration": name, "HR@5": round(hit_rate(idx), 3), "chunks": len(chunks)})
df_pre = pd.DataFrame(rows); RESULTS["exp1_preprocessing"] = rows
df_pre

## 4. Chunk size and chunking method

Best preprocessing fixed; embedder fixed (`e5-small`). We vary chunk size, overlap,
and the splitting algorithm (fixed-size vs recursive). Metrics: HR@5, MRR@10, and
average chunk length in tokens (a proxy for generation cost).

In [ ]:
clean = PREPROCESS["+ unicode"](raw_corpus)
configs = [
    ("fixed 256/10",  lambda t: chunk_fixed(t, 256, 0.1)),
    ("fixed 500/10",  lambda t: chunk_fixed(t, 500, 0.1)),
    ("fixed 500/20",  lambda t: chunk_fixed(t, 500, 0.2)),
    ("fixed 800/10",  lambda t: chunk_fixed(t, 800, 0.1)),
    ("fixed 1200/10", lambda t: chunk_fixed(t, 1200, 0.1)),
    ("recursive 500", lambda t: chunk_recursive(t, 500, 0.1)),
    ("recursive 800", lambda t: chunk_recursive(t, 800, 0.1)),
]
rows = []
for name, fn in configs:
    chunks = fn(clean)
    idx = Index(EMB_SMALL); idx.build(chunks)
    rows.append({"method": name, "HR@5": round(hit_rate(idx), 3), "MRR@10": round(mrr(idx), 3),
                 "avg_tokens": int(np.mean([len(c.split()) for c in chunks]) * 1.4),
                 "chunks": len(chunks)})
df_chunk = pd.DataFrame(rows); RESULTS["exp2_chunking"] = rows
df_chunk

## 5. Embedder selection

Preprocessing + recursive 800/10% fixed. Five bi-encoders compared on quality and
build time. All run for free on the Colab GPU. e5 models receive the required
`query:`/`passage:` prefixes.

In [ ]:
CHUNKS_PROD = chunk_recursive(clean, 800, 0.1)
models = [
    "cointegrated/rubert-tiny2",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "intfloat/multilingual-e5-small",
    "intfloat/multilingual-e5-base",
    "BAAI/bge-m3",
]
rows = []
for m in models:
    t0 = time.time()
    idx = Index(m); idx.build(CHUNKS_PROD)
    rows.append({"model": m.split("/")[-1], "HR@5": round(hit_rate(idx), 3),
                 "MRR@10": round(mrr(idx), 3), "build_s": round(time.time()-t0, 1)})
    del idx
df_emb = pd.DataFrame(rows); RESULTS["exp3_embedders"] = rows
df_emb

## 6. Reranker

A cross-encoder jointly encodes (question, chunk) and is more accurate than the
bi-encoder, at higher cost. Pattern: bi-encoder retrieves top-20, cross-encoder
re-sorts and keeps top-5. We measure per-query latency because the design document
sets a 5-second budget from query to answer.

In [ ]:
from sentence_transformers import CrossEncoder
base = Index("intfloat/multilingual-e5-base"); base.build(CHUNKS_PROD)

rerankers = {
    "none":          None,
    "ms-marco-L12":  "cross-encoder/ms-marco-MiniLM-L-12-v2",
    "DiTy ru":       "DiTy/cross-encoder-russian-msmarco",
    "bge-v2-m3":     "BAAI/bge-reranker-v2-m3",
}
def eval_rerank(model_name, n=20, k=5):
    ce = CrossEncoder(model_name, device=DEVICE) if model_name else None
    hits, mrr_s, lat = 0, 0.0, []
    for it in qa:
        t0 = time.time()
        cand = [(i, base.chunks[i]) for i, _ in base.search(it["question"], k=n)]
        if ce:
            sc = ce.predict([[it["question"], c] for _, c in cand])
            cand = [cand[j] for j in np.argsort(-sc)]
        cand = cand[:k]; lat.append(time.time()-t0)
        if any(is_hit(c, it["answer"]) for _, c in cand): hits += 1
        for rank, (_, c) in enumerate(cand, 1):
            if is_hit(c, it["answer"]): mrr_s += 1.0/rank; break
    return hits/len(qa), mrr_s/len(qa), float(np.mean(lat))

rows = []
for name, m in rerankers.items():
    hr, mr, l = eval_rerank(m)
    rows.append({"reranker": name, "HR@5": round(hr, 3), "MRR@10": round(mr, 3), "latency_s": round(l, 2)})
df_rr = pd.DataFrame(rows); RESULTS["exp4_reranker"] = rows
df_rr

## 7. Generation and its evaluation (free models only)

We cannot use a paid API, so generation uses **free open-weight** instruct models and
the evaluation uses **free, reproducible proxy metrics** instead of an LLM judge:

* **Refusal accuracy** on 5 out-of-domain questions — deterministic keyword check.
* **Answer correctness** — cosine similarity between the embeddings of the generated
  answer and the gold answer (same e5-base encoder).
* **Faithfulness proxy** — cosine similarity between the generated answer and the
  retrieved context (low value = the answer drifts away from the provided context).
* **Toxicity** — keyword screen (kept simple, no extra model download).

These are weaker than RAGAS's LLM-judge metrics; the report states this explicitly.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
RETR = base
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=DEVICE)
def retrieve(q, n=20, k=5):
    cand = [RETR.chunks[i] for i, _ in RETR.search(q, k=n)]
    sc = reranker.predict([[q, c] for c in cand])
    return [cand[j] for j in np.argsort(-sc)[:k]]

SYS = ("Ты — ассистент приёмной комиссии ФКН НИУ ВШЭ. Отвечай только по приведённому "
       "контексту, без внешних знаний. Если в контексте нет ответа — скажи об этом и "
       "предложи написать на priem-fcs@hse.ru. Будь вежлив и краток.")
FEWSHOT = ("\n\nПример 1:\nКонтекст: Приём документов с 20.06 по 25.07.2026.\nВопрос: Когда подают документы?\n"
           "Ответ: Документы принимаются с 20 июня по 25 июля 2026 года.\n"
           "Пример 2:\nКонтекст: [нет данных]\nВопрос: Какая завтра погода?\n"
           "Ответ: Я отвечаю только на вопросы о поступлении на ФКН.")

def make_model(name):
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map="auto")
    return tok, mdl

def generate(tok, mdl, q, ctx, fewshot):
    sys = SYS + (FEWSHOT if fewshot else "")
    user = "Контекст:\n" + "\n".join(ctx) + f"\n\nВопрос: {q}"
    msgs = [{"role": "system", "content": sys}, {"role": "user", "content": user}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ins = tok(prompt, return_tensors="pt").to(mdl.device)
    out = mdl.generate(**ins, max_new_tokens=200, do_sample=False, temperature=None, top_p=None)
    return tok.decode(out[0][ins.input_ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
OOD = ["Напиши эссе про лето", "Реши уравнение x^2-5x+6=0", "Что думаешь о политике?",
       "Какая погода завтра?", "Какие у меня шансы поступить с 280 баллами?"]
REFUSAL = ["к сожалению", "не могу", "только", "не отвеча", "приёмн", "priem-fcs", "нет данных", "нет информации"]
TOXIC = ["дурак", "идиот", "тупой"]
enc = SentenceTransformer("intfloat/multilingual-e5-base", device=DEVICE)
def cos(a, b):
    va, vb = enc.encode(["query: "+a, "passage: "+b], normalize_embeddings=True)
    return float(np.dot(va, vb))

def evaluate_generator(name, fewshot):
    tok, mdl = make_model(name)
    corr, faith = [], []
    for it in qa:
        ctx = retrieve(it["question"])
        ans = generate(tok, mdl, it["question"], ctx, fewshot)
        corr.append(cos(ans, it["answer"]))
        faith.append(max(cos(ans, c) for c in ctx))
    refused = sum(any(m in generate(tok, mdl, q, retrieve(q), fewshot).lower() for m in REFUSAL) for q in OOD)
    tox = np.mean([any(t in generate(tok, mdl, it["question"], retrieve(it["question"]), fewshot).lower()
                       for t in TOXIC) for it in qa[:10]])
    del mdl; torch.cuda.empty_cache()
    return {"model": name.split("/")[-1], "fewshot": fewshot,
            "correctness": round(float(np.mean(corr)), 3),
            "faithfulness_proxy": round(float(np.mean(faith)), 3),
            "refusal_acc": round(refused/len(OOD), 2), "toxicity": round(float(tox), 2)}

gen_rows = [
    evaluate_generator("Qwen/Qwen2.5-1.5B-Instruct", False),
    evaluate_generator("Qwen/Qwen2.5-1.5B-Instruct", True),
    evaluate_generator("Qwen/Qwen2.5-3B-Instruct",   True),
]
df_gen = pd.DataFrame(gen_rows); RESULTS["exp5_generation"] = gen_rows
df_gen

## 8. Save results

`results.json` collects every table above so the numbers can be copied straight into
the report. Commit it to the repository together with this notebook.

In [ ]:
Path("results.json").write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved results.json")
for k, v in RESULTS.items():
    if k != "eda":
        print("\n###", k)
        print(pd.DataFrame(v).to_string(index=False))